In [21]:
import imageio
import numpy as np
from utilities.dataset import BasicDataset
import torch
from utilities.utils import get_model
from simple_camera_pipeline.python.pipeline import run_pipeline_v2
from simple_camera_pipeline.python.pipeline_utils import get_metadata
from fractions import Fraction
import os

In [22]:
class FakeArgs:
    def __init__(self):
        self.width = 128
        self.depth = 152
        self.down = 4
        self.model = "isotropic"

class FakeESUMArgs:
    def __init__(self):
        self.model = "simp_multi"

def write_im(fpath,tensor):
    tensor = tensor.squeeze().permute(1,2,0).cpu().numpy()
    tensor = run_pipeline_v2((tensor+1)/2,{'input_stage':'vignetting_correction','output_stage':'tone'},get_metadata('DSC01201_PSMS.dng'))
    tensor = (tensor*255).astype(np.uint8)
    imageio.imwrite(fpath,tensor)

def fix_state_dict(state_dict):
    new_dict = dict()
    for key in state_dict:
        new_dict[key[7:]]=state_dict[key]
    return new_dict

dataset = BasicDataset("../scenes","test","single_quad_nona",False)
print(len(dataset))
iso_net = get_model(FakeArgs(),"cuda")[0].eval().cuda()
iso_net.load_state_dict(fix_state_dict(torch.load('../models/isotropic-64-12/net.pth')['net_state_dict']))
esum_net = get_model(FakeESUMArgs(),"cuda")[0].eval().cuda()
esum_net.load_state_dict(fix_state_dict(torch.load('models/iso3200.pth')['net_state_dict']))

8
() {'width': 128, 'depth': 152, 'downsample': 4, 'drop_path_rate': 0}


<All keys matched successfully>

In [23]:
for data in dataset:
    with torch.no_grad():
        base_path = data['img_file'].replace('scenes','vis').replace('.npy','')
        if (os.path.exists(base_path+'_gt.png')):
            continue
        img = data['image'].unsqueeze(0).cuda()
        iso_single,iso_quad,iso_nona = iso_net(img)
        esum_single,esum_quad,esum_nona = esum_net(img)
        write_im(base_path+'_gt.png',data['gt'])
        write_im(base_path+'_iso_nona.png',iso_nona)
        write_im(base_path+'_esum_nona.png',esum_nona)

CFAPattern is None; using [0, 1, 1, 2] (RGGB)
CFAPattern is None; using [0, 1, 1, 2] (RGGB)
CFAPattern is None; using [0, 1, 1, 2] (RGGB)
CFAPattern is None; using [0, 1, 1, 2] (RGGB)
CFAPattern is None; using [0, 1, 1, 2] (RGGB)
CFAPattern is None; using [0, 1, 1, 2] (RGGB)
